# SDS Detection Demonstration

This notebook demonstrates how the ProcessBehavior library detects Sampling Design States (SDS 0-6) using synthetic data generators.

## What are Sampling Design States?

SDS classification (by Wheeler/Bishop) determines:
- What charts are appropriate for your data
- How to calculate variance (within-cell vs moving average)
- Whether interactions can be analyzed

**Key Concept**: Cells are defined by **subgroups** (grouping_vars), not subgroup×time combinations.
- Sample size n = observations per subgroup **across all time points**
- time_var is for sequencing/plotting, not cell definition

In [1]:
# Import required modules
import pandas as pd
from processbehavior import ProcessBehavior
from processbehavior.datasets import synthetic

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

print("Imports successful")

Imports successful


---

## SDS 0: No Structure

**Characteristics**:
- No grouping variables OR no time variable
- Simple Individual-Moving Range (IMR) chart
- Data ordered by obs_id

In [2]:
print("=" * 80)
print("SDS 0: NO STRUCTURE")
print("=" * 80)

# Create simple data with no grouping or time structure
df_sds0 = pd.DataFrame({
    'y': [50.1, 49.8, 50.3, 49.7, 50.0, 50.2, 49.9, 50.1] * 3
})

print(f"\nData: {len(df_sds0)} observations, no grouping or time variables")
print(df_sds0.head(10))

# Analyze using new API
pb_sds0 = ProcessBehavior(df_sds0)
study_sds0 = pb_sds0.formulate(
    response=pb_sds0.cols.y
    # No factors, no time
)
result_sds0 = study_sds0.execute()

print(f"\n{'='*80}")
print(f"Detected SDS: {study_sds0.sds}")
print(f"Description: {study_sds0.sds_description}")
print(f"Charts available: {result_sds0.all_charts}")
print(f"Expected: SDS 0, IMR chart only")
print(f"{'='*80}\n")

SDS 0: NO STRUCTURE

Data: 24 observations, no grouping or time variables
      y
0  50.1
1  49.8
2  50.3
3  49.7
4  50.0
5  50.2
6  49.9
7  50.1
8  50.1
9  49.8

Detected SDS: 0
Description: Individual measurements with no rational subgrouping or time structure
Charts available: ['all']
Expected: SDS 0, IMR chart only



---

## SDS 1: Full Replication

**Characteristics**:
- Every subgroup has n ≥ 2 observations
- True within-subgroup variance can be estimated
- Most statistically powerful
- Supports Xbar AND S charts

In [3]:
print("=" * 80)
print("SDS 1: FULL REPLICATION")
print("=" * 80)

# Generate SDS 1 data: K=3 factors, T=8 times, n=2-4 per subgroup
df_sds1 = synthetic.make_sds1(
    K=3, 
    T=8, 
    n_min=2, 
    n_max=4,
    seed=42
)

print(f"\nData: {len(df_sds1)} observations")
print(df_sds1.head(10))

# Check sample sizes
cell_counts = df_sds1.groupby(['factor 1', 'time']).size()
print(f"\nSubgroup×Time combinations: {len(cell_counts)}")
print(f"Sample size per cell: min={cell_counts.min()}, max={cell_counts.max()}")
print(f"All cells have n≥2: {(cell_counts >= 2).all()}")

# Sample sizes per subgroup (across all time)
subgroup_sizes = df_sds1.groupby('factor 1').size()
print(f"\nSubgroup sizes (across all time):")
print(subgroup_sizes)

# Analyze using new API
pb_sds1 = ProcessBehavior(df_sds1)
study_sds1 = pb_sds1.formulate(
    response='y',
    factors=['factor 1'],
    time='time'
)
result_sds1 = study_sds1.execute()

print(f"\n{'='*80}")
print(f"Detected SDS: {study_sds1.sds}")
print(f"Description: {study_sds1.sds_description}")
print(f"Charts available: {result_sds1.all_charts}")
print(f"Expected: SDS 1, both Xbar AND Sbar charts available")
print(f"Has S Chart: {'Sbar' in result_sds1.all_charts}")
print(f"{'='*80}\n")

SDS 1: FULL REPLICATION

Data: 76 observations
   time factor 1 factor 2          y
0     1       K1       NA  51.893316
1     1       K1       NA  51.602832
2     1       K1       NA  51.609102
3     1       K1       NA  52.199132
4     2       K1       NA  48.988716
5     2       K1       NA  48.908676
6     3       K1       NA  49.963740
7     3       K1       NA  49.917550
8     3       K1       NA  49.958351
9     3       K1       NA  50.219447

Subgroup×Time combinations: 24
Sample size per cell: min=2, max=4
All cells have n≥2: True

Subgroup sizes (across all time):
factor 1
K1    27
K2    23
K3    26
dtype: int64

Detected SDS: 1
Description: All factor × time cells have n ≥ 2 observations (best case for analysis)
Charts available: ['Xbar', 'Sbar']
Expected: SDS 1, both Xbar AND Sbar charts available
Has S Chart: True



---

## SDS 2: No Replication

**Characteristics**:
- Exactly n=1 observation per (factor×time) cell
- Complete factorial grid
- Must use moving average for variance estimation
- Only Xbar charts (no S charts)

In [4]:
print("=" * 80)
print("SDS 2: NO REPLICATION")
print("=" * 80)

# Generate SDS 2 data: K=3 factors, T=10 times, exactly n=1 per cell
df_sds2 = synthetic.make_sds2(
    K=3,
    T=10,
    seed=42
)

print(f"\nData: {len(df_sds2)} observations")
print(df_sds2.head(10))

# Check sample sizes
cell_counts = df_sds2.groupby(['factor 1', 'time']).size()
print(f"\nSubgroup×Time combinations: {len(cell_counts)}")
print(f"Sample size per cell: min={cell_counts.min()}, max={cell_counts.max()}")
print(f"All cells have exactly n=1: {(cell_counts == 1).all()}")

# Expected: 3 factors × 10 times = 30 cells
print(f"Expected cells (K×T): {3 * 10}")
print(f"Actual cells: {len(cell_counts)}")

# Analyze using new API
pb_sds2 = ProcessBehavior(df_sds2)
study_sds2 = pb_sds2.formulate(
    response='y',
    factors=['factor 1'],
    time='time'
)
result_sds2 = study_sds2.execute()

print(f"\n{'='*80}")
print(f"Detected SDS: {study_sds2.sds}")
print(f"Description: {study_sds2.sds_description}")
print(f"Charts available: {result_sds2.all_charts}")
print(f"Expected: SDS 2, Xbar only (no S charts)")
print(f"S charts NOT available: {'Sbar' not in result_sds2.all_charts}")
print(f"{'='*80}\n")

SDS 2: NO REPLICATION

Data: 30 observations
   time factor 1 factor 2          y
0     1       K1       NA  52.507321
1     2       K1       NA  48.595372
2     3       K1       NA  48.618719
3     4       K1       NA  51.332665
4     5       K1       NA  49.744052
5     6       K1       NA  51.387908
6     7       K1       NA  49.582858
7     8       K1       NA  51.669442
8     9       K1       NA  51.386742
9    10       K1       NA  50.839333

Subgroup×Time combinations: 30
Sample size per cell: min=1, max=1
All cells have exactly n=1: True
Expected cells (K×T): 30
Actual cells: 30

Detected SDS: 1
Description: All factor × time cells have n ≥ 2 observations (best case for analysis)
Charts available: ['Xbar', 'Sbar']
Expected: SDS 2, Xbar only (no S charts)
S charts NOT available: False



---

## SDS 3: Partial Replication (MOST COMMON!)

**Characteristics**:
- Some cells have n=1, others have n≥2
- Requires hybrid variance estimation
- Most common in real-world data
- Most challenging to analyze correctly

In [5]:
print("=" * 80)
print("SDS 3: PARTIAL REPLICATION")
print("=" * 80)

# Generate SDS 3 data: K=3 factors, T=8 times, 50% cells replicated
df_sds3 = synthetic.make_sds3(
    K=3,
    T=8,
    p_replicated=0.5,
    n_when_replicated=3,
    seed=42
)

print(f"\nData: {len(df_sds3)} observations")
print(df_sds3.head(10))

# Check sample sizes
cell_counts = df_sds3.groupby(['factor 1', 'time']).size()
print(f"\nSubgroup×Time combinations: {len(cell_counts)}")
print(f"Sample size per cell: min={cell_counts.min()}, max={cell_counts.max()}")
print(f"Cells with n=1: {(cell_counts == 1).sum()}")
print(f"Cells with n≥2: {(cell_counts >= 2).sum()}")

# Analyze using new API
pb_sds3 = ProcessBehavior(df_sds3)
study_sds3 = pb_sds3.formulate(
    response='y',
    factors=['factor 1'],
    time='time'
)
result_sds3 = study_sds3.execute()

print(f"\n{'='*80}")
print(f"Detected SDS: {study_sds3.sds}")
print(f"Description: {study_sds3.sds_description}")
print(f"Charts available: {result_sds3.all_charts}")
print(f"Expected: SDS 3, hybrid handling")
print(f"{'='*80}\n")

SDS 3: PARTIAL REPLICATION

Data: 48 observations
   time factor 1 factor 2          y     cell_type
0     1       K1       NA  51.881921    replicated
1     1       K1       NA  51.518817    replicated
2     1       K1       NA  51.526654    replicated
3     2       K1       NA  49.063041  unreplicated
4     3       K1       NA  49.285278  unreplicated
5     4       K1       NA  50.638274    replicated
6     4       K1       NA  51.087110    replicated
7     4       K1       NA  51.029372    replicated
8     5       K1       NA  50.299260    replicated
9     5       K1       NA  49.975343    replicated

Subgroup×Time combinations: 24
Sample size per cell: min=1, max=3
Cells with n=1: 12
Cells with n≥2: 12

Detected SDS: 1
Description: All factor × time cells have n ≥ 2 observations (best case for analysis)
Charts available: ['Xbar', 'Sbar']
Expected: SDS 3, hybrid handling



---

## SDS 4: Single Condition Over Time

**Characteristics**:
- Only one factor level (K=1)
- Multiple time points
- Classic time series / IMR chart scenario
- May show trends or drift

In [6]:
print("=" * 80)
print("SDS 4: SINGLE CONDITION OVER TIME")
print("=" * 80)

# Generate SDS 4 data: T=50 time points, single factor
df_sds4 = synthetic.make_sds4(
    T=50,
    drift_type='random_walk',
    seed=42
)

print(f"\nData: {len(df_sds4)} observations")
print(df_sds4.head(10))
print(f"\nFactor levels: {df_sds4['factor 1'].unique()}")
print(f"Time points: {df_sds4['time'].nunique()}")

# Analyze using new API
pb_sds4 = ProcessBehavior(df_sds4)
study_sds4 = pb_sds4.formulate(
    response='y',
    factors=['factor 1'],
    time='time'
)
result_sds4 = study_sds4.execute()

print(f"\n{'='*80}")
print(f"Detected SDS: {study_sds4.sds}")
print(f"Description: {study_sds4.sds_description}")
print(f"Charts available: {result_sds4.all_charts}")
print(f"Expected: SDS 4, IMR chart for time series")
print(f"{'='*80}\n")

SDS 4 detected: Single condition over time.
Xbar analysis may not be appropriate.
Consider using 'Imr' analysis.


SDS 4: SINGLE CONDITION OVER TIME

Data: 50 observations
   time factor 1 factor 2          y
0     1       K1       NA  50.161355
1     2       K1       NA  50.142225
2     3       K1       NA  49.419415
3     4       K1       NA  50.015494
4     5       K1       NA  49.662558
5     6       K1       NA  49.399829
6     7       K1       NA  49.564499
7     8       K1       NA  50.225096
8     9       K1       NA  49.278267
9    10       K1       NA  49.883954

Factor levels: ['K1']
Time points: 50

Detected SDS: 4
Description: Grouping factors present but no time variable
Charts available: ['Xbar', 'Sbar']
Expected: SDS 4, IMR chart for time series



---

## SDS 5: Nested Design

**Characteristics**:
- Hierarchical structure (Factor 2 nested within Factor 1)
- Asynchronous operation (not all combinations at all times)
- Common in multi-head machines
- Requires variance component analysis

In [7]:
print("=" * 80)
print("SDS 5: NESTED DESIGN")
print("=" * 80)

# Generate SDS 5 data: L=2 lines, H_per_L=3 heads per line, T=8 times
df_sds5 = synthetic.make_sds5(
    L=2,
    H_per_L=3,
    T=8,
    p_active=0.8,
    seed=42
)

print(f"\nData: {len(df_sds5)} observations")
print(df_sds5.head(10))

# Verify nesting structure
nesting_check = df_sds5.groupby('factor 2')['factor 1'].nunique()
print(f"\nNesting verification (each head belongs to one line):")
print(nesting_check)
print(f"All heads nested correctly: {(nesting_check == 1).all()}")

# Check coverage
full_grid = 2 * 3 * 8  # L × H_per_L × T
actual_cells = df_sds5.groupby(['factor 1', 'factor 2', 'time']).size().shape[0]
print(f"\nFull grid size: {full_grid}")
print(f"Actual cells: {actual_cells} ({actual_cells/full_grid:.1%} coverage)")
print(f"Asynchronous (incomplete grid): {actual_cells < full_grid * 0.95}")

# Analyze using new API
pb_sds5 = ProcessBehavior(df_sds5)
study_sds5 = pb_sds5.formulate(
    response='y',
    factors=['factor 1', 'factor 2'],
    time='time'
)
result_sds5 = study_sds5.execute()

print(f"\n{'='*80}")
print(f"Detected SDS: {study_sds5.sds}")
print(f"Description: {study_sds5.sds_description}")
print(f"Charts available: {result_sds5.all_charts}")
print(f"Expected: SDS 5, nested structure with asynchronous coverage")
print(f"{'='*80}\n")

SDS 5: NESTED DESIGN

Data: 41 observations
   time factor 1     factor 2          y
0     1    Line1  Line1_Head1  51.175113
1     2    Line1  Line1_Head1  50.536597
2     4    Line1  Line1_Head1  52.195043
3     5    Line1  Line1_Head1  51.558888
4     6    Line1  Line1_Head1  52.426771
5     1    Line1  Line1_Head2  51.200495
6     2    Line1  Line1_Head2  50.537771
7     4    Line1  Line1_Head2  52.432470
8     5    Line1  Line1_Head2  51.900125
9     6    Line1  Line1_Head2  52.669054

Nesting verification (each head belongs to one line):
factor 2
Line1_Head1    1
Line1_Head2    1
Line1_Head3    1
Line2_Head1    1
Line2_Head2    1
Line2_Head3    1
Name: factor 1, dtype: int64
All heads nested correctly: True

Full grid size: 48
Actual cells: 41 (85.4% coverage)
Asynchronous (incomplete grid): True

Detected SDS: 1
Description: All factor × time cells have n ≥ 2 observations (best case for analysis)
Charts available: ['Xbar', 'Sbar']
Expected: SDS 5, nested structure with asynchron

---

## SDS 6: Unstructured / Regime Changes

**Characteristics**:
- Incomplete (factor×time) grid
- Process regime changes (mean shifts over time)
- Irregular sampling patterns
- Common in long-term studies with adjustments

In [8]:
print("=" * 80)
print("SDS 6: UNSTRUCTURED / REGIME CHANGES")
print("=" * 80)

# Generate SDS 6 data: T=80 times, K=3 factors, sparse sampling
df_sds6 = synthetic.make_sds6(
    T=80,
    K=3,
    p_sampled=0.6,  # Only 60% of slots sampled
    seed=42
)

print(f"\nData: {len(df_sds6)} observations")
print(df_sds6.head(10))

# Check coverage
full_grid = 3 * 80  # K × T
actual_cells = df_sds6.groupby(['factor 1', 'time']).size().shape[0]
print(f"\nFull grid size (K×T): {full_grid}")
print(f"Actual cells: {actual_cells} ({actual_cells/full_grid:.1%} coverage)")
print(f"Incomplete grid (< 75%): {actual_cells / full_grid < 0.75}")

# Check regime distribution
print(f"\nRegime distribution:")
print(df_sds6.groupby('regime').size())

# Analyze using new API
pb_sds6 = ProcessBehavior(df_sds6)
study_sds6 = pb_sds6.formulate(
    response='y',
    factors=['factor 1'],
    time='time'
)
result_sds6 = study_sds6.execute()

print(f"\n{'='*80}")
print(f"Detected SDS: {study_sds6.sds}")
print(f"Description: {study_sds6.sds_description}")
print(f"Charts available: {result_sds6.all_charts}")
print(f"Expected: SDS 6, irregular pattern with incomplete grid")
print(f"{'='*80}\n")

SDS 6: UNSTRUCTURED / REGIME CHANGES

Data: 153 observations
   time  factor 1 factor 2          y  regime
0     1  Machine2       NA  47.191949       0
1     2  Machine2       NA  47.567728       0
2     2  Machine3       NA  50.739708       0
3     3  Machine3       NA  49.921166       0
4     4  Machine1       NA  49.523528       0
5     4  Machine2       NA  47.035597       0
6     5  Machine1       NA  49.334327       0
7     6  Machine1       NA  49.754857       0
8     6  Machine2       NA  47.343439       0
9     6  Machine3       NA  51.421636       0

Full grid size (K×T): 240
Actual cells: 153 (63.7% coverage)
Incomplete grid (< 75%): True

Regime distribution:
regime
0    37
1    38
2    42
3    36
dtype: int64



Detected SDS: 1
Description: All factor × time cells have n ≥ 2 observations (best case for analysis)
Charts available: ['Xbar', 'Sbar']
Expected: SDS 6, irregular pattern with incomplete grid



---

## Summary Comparison Table

Compare all SDS classifications and their characteristics:

In [9]:
# Create summary comparison
summary_data = [
    {
        'SDS': 0,
        'Detected': study_sds0.sds,
        'Description': 'No structure',
        'Charts': ', '.join(result_sds0.all_charts),
        'n_obs': len(df_sds0)
    },
    {
        'SDS': 1,
        'Detected': study_sds1.sds,
        'Description': 'Full replication',
        'Charts': ', '.join(result_sds1.all_charts),
        'n_obs': len(df_sds1),
        'Has_S_Chart': 'Sbar' in result_sds1.all_charts
    },
    {
        'SDS': 2,
        'Detected': study_sds2.sds,
        'Description': 'No replication',
        'Charts': ', '.join(result_sds2.all_charts),
        'n_obs': len(df_sds2),
        'Has_S_Chart': 'Sbar' in result_sds2.all_charts
    },
    {
        'SDS': 3,
        'Detected': study_sds3.sds,
        'Description': 'Partial replication',
        'Charts': ', '.join(result_sds3.all_charts),
        'n_obs': len(df_sds3),
        'Has_S_Chart': 'Sbar' in result_sds3.all_charts
    },
    {
        'SDS': 4,
        'Detected': study_sds4.sds,
        'Description': 'Time series',
        'Charts': ', '.join(result_sds4.all_charts),
        'n_obs': len(df_sds4)
    },
    {
        'SDS': 5,
        'Detected': study_sds5.sds,
        'Description': 'Nested design',
        'Charts': ', '.join(result_sds5.all_charts),
        'n_obs': len(df_sds5)
    },
    {
        'SDS': 6,
        'Detected': study_sds6.sds,
        'Description': 'Regime changes',
        'Charts': ', '.join(result_sds6.all_charts),
        'n_obs': len(df_sds6)
    }
]

summary_df = pd.DataFrame(summary_data)
print("\n" + "=" * 100)
print("SDS DETECTION SUMMARY")
print("=" * 100)
print(summary_df.to_string(index=False))

# Validation check
all_correct = all(row['SDS'] == row['Detected'] for row in summary_data)
print(f"\n{'='*100}")
if all_correct:
    print("ALL SDS TYPES CORRECTLY DETECTED!")
else:
    print("Some misclassifications detected")
    mismatches = [row for row in summary_data if row['SDS'] != row['Detected']]
    for m in mismatches:
        print(f"  - Expected SDS {m['SDS']}, got SDS {m['Detected']}")
print(f"{'='*100}\n")


SDS DETECTION SUMMARY
 SDS  Detected         Description     Charts  n_obs Has_S_Chart
   0         0        No structure        all     24         NaN
   1         1    Full replication Xbar, Sbar     76        True
   2         1      No replication Xbar, Sbar     30        True
   3         1 Partial replication Xbar, Sbar     48        True
   4         4         Time series Xbar, Sbar     50         NaN
   5         1       Nested design Xbar, Sbar     41         NaN
   6         1      Regime changes Xbar, Sbar    153         NaN

Some misclassifications detected
  - Expected SDS 2, got SDS 1
  - Expected SDS 3, got SDS 1
  - Expected SDS 5, got SDS 1
  - Expected SDS 6, got SDS 1



---

## Key Takeaways

1. **Cells = Subgroups**: For SDS detection, cells are defined by grouping_vars ONLY
   - NOT by (grouping_vars × time_var) combinations
   - Sample size n = observations per subgroup across ALL time points

2. **time_var Purpose**: Used for sequencing and plotting, not cell definition
   - Enables proper time series ordering
   - Allows stratified IMR charts
   - Does not fragment subgroups into separate cells

3. **Chart Availability**:
   - SDS 1: Xbar AND S charts (full replication)
   - SDS 2: Xbar only (no S charts, n=1 per cell)
   - SDS 3: Hybrid (depends on replication pattern)
   - SDS 4: IMR (single condition time series)

4. **Why This Matters**:
   - Incorrect SDS detection → wrong charts → invalid conclusions
   - Treating time as part of cell definition incorrectly fragments data
   - Example: 100 observations per subgroup over 100 time points:
     - ✓ Correct: SDS 1 (n=100 per subgroup) → S charts available
     - ✗ Wrong: SDS 2 (n=1 per subgroup×time cell) → no S charts

---

## Next Steps

- Try modifying the generators to create edge cases
- Experiment with different K, T, and replication parameters
- Review the synthetic.py documentation for all generator options
- Apply these patterns to your real data